# Pydantic AI Lab - Agents, Tools, and MCP

A reference notebook for building a Pydantic AI agent from a simple model call into a worker that plans and completes a task using a SQLite todo board and filesystem tools.

| Section | What it demonstrates |
|---|---|
| Setup | Load environment settings and import Pydantic AI / MCP APIs |
| Steps 1-2 | Configure an agent, run it asynchronously, and inspect its output |
| Todo board | Seed and inspect a shared SQLite task board |
| Step 3 | Define Python functions that can be exposed as agent tools |
| Step 4 | Connect a filesystem MCP server scoped to a workspace folder |
| Step 5 | Combine the board tools and filesystem tools in a goal-driven worker |

**Prerequisites:** an API key available through `.env` or the environment; the project's `board.py`; the notebook dependencies (`pydantic-ai`, `python-dotenv`, and FastMCP); and `npx` for launching `@modelcontextprotocol/server-filesystem`.

**Important side effects:** `load_dotenv(override=True)` allows `.env` values to replace existing environment values. The board-seeding cells call `board.reset_board()`, which drops and recreates the SQLite `todos` table. The final worker calls an LLM and writes a translation file through the filesystem tools. Read the goal and workspace setup before rerunning those cells.

**Execution context:** `Path("workspace").resolve()` is relative to the notebook kernel's current working directory, so run from this project directory for the intended workspace path.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

# Load .env values; override=True lets them replace matching existing environment variables.
load_dotenv(override=True)

True

## Step 1: Create The Agent

The model identifier is supplied as the agent's positional model argument. `instructions` set the agent's behavior; creating the `Agent` configures it but does not make a model request yet.

In [4]:
MODEL = 'openai-chat:gpt-5.4-mini' 

agent = Agent(
    MODEL, 
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)



## Step 2 Run It

`await agent.run(...)` sends the prompt and waits for the response. `result.output` is the user-facing result for this run; top-level `await` is supported in notebook cells.

In [5]:
result = await agent.run("Say hello in Tagalog")
print(result.output)

Kamusta!


# Project: SQLite Todo Board

The helper module `board.py` persists goals and steps in SQLite. This cell clears the existing board and inserts one new top-level goal, then prints the stored rows. **Rerunning `board.reset_board()` discards the board's previous todos.**

In [6]:
import board

# CAUTION: reset_board() drops and recreates the SQLite todos table.
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt',
  'status': 'pending',
  'result': ''}]

`show_board()` renders the same board for a person using Rich formatting; it does not add or update tasks.

In [7]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt

## Step 3: Add Tools

These functions are the board operations the model can request. Type annotations and docstrings describe each tool's inputs and purpose. `plan_steps` creates child todos and returns their IDs. In this introductory `board_agent`, only `show_todos` and `complete_task` are registered; planning is added to the full worker in Step 5.

In [8]:
def show_todos() ->list[dict]:
    """ List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """ Break a goal into a ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {
        "goal_id": goal_id, 
        "steps_ids":[board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """ Mark a todo (a step or the goal) with this id as done and record a short result summary. """
    board.complete_todo(task_id, result)
    return {
        'task_id': task_id,
        'status': 'completed'
    }

In [9]:
board_agent = Agent(
    MODEL,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [10]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.output)

There is 1 item on the board:

- **ID 1** — “Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt”
  - **Status:** pending




## Step 4: Add MCP

`MCPToolset` starts the filesystem MCP server over standard I/O. The resolved `workspace` path is passed as the server's allowed root; the server's tools are then attached to `file_agent`. The path is relative to the kernel's current working directory.

In [11]:
# Resolve the allowed filesystem root relative to the notebook kernel working directory.
workspace = Path("workspace").resolve()

filesystem = MCPToolset(
    StdioTransport(
        command='npx',
        args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
        cwd=str(workspace), 
        log_file=Path(os.devnull),
    ),
    init_timeout=60
)

In [12]:
file_agent = Agent(
    MODEL,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    toolsets=[filesystem]
)

In [13]:
async with file_agent:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
    print(result.output)

The notes welcome the team and explain that you’re building a small language tutor one task at a time.


## Step 5: Put It In A Loop With A Goal

This final stage combines the board tools (list, plan, complete) with the filesystem MCP toolset. The cell resets and seeds the board, claims the new goal, asks the worker to act, and displays the resulting board. The saved output shows that the goal and its three steps were marked complete; no cells were rerun as part of this review.

In [14]:
INSTRUCTIONS =""" 
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through Begin by laying out a short plan: 
the handful of concrete steps the work itself breaks down into added to the
board under the goal. Then carry them out with your file tools, marking each
step completed as you finish it. Once the steps are all done, close the goal.
Your files live in the single your tools are allowed to use. 
"""

worker = Agent(
    MODEL,
    instructions=INSTRUCTIONS,
    tools=[show_todos, 
           plan_steps, 
           complete_task],
    toolsets=[filesystem]
)

# This seed reset is destructive: it removes existing board todos.
board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contented into Tagalog, and write the Tagalog to tagalog.txt.")
# This changes the goal to in_progress before the worker starts.
board.claim_todo(goal_id)

async with worker:
    result = await worker.run("Please work the pending goal on the board.")

print(result.output)
board.show_board()

Done. I read `notes.txt`, translated it into Tagalog, and wrote the result to `tagalog.txt`.


Goal #1: Read notes.txt, translate its contented into Tagalog, and write the Tagalog to tagalog.txt.  Completed reading, translating, and writing the Tagalog file.
  Step #2: Read notes.txt  Read notes.txt successfully.
  Step #3: Translate the contents into Tagalog  Translated the notes into Tagalog.
  Step #4: Write the Tagalog translation to tagalog.txt  Wrote the Tagalog translation to tagalog.txt.

## Conclusion and Quick Reference

**Core pattern:** configure an `Agent` with a model and instructions; use `await agent.run(...)` to run it; expose Python functions through `tools`; connect external capabilities with `toolsets`; then combine these in a worker loop that plans, performs, and records task progress.

- `board.py` is the persistent SQLite task store; `show_board()` is only a formatted view.
- `tools=[...]` controls which board functions a particular agent can call. The Step 3 agent is deliberately narrower than the Step 5 worker.
- The filesystem MCP server is launched with the resolved workspace as its allowed directory. Keep the notebook's working directory in mind when resolving relative paths.
- Run the setup and task cells in order. Board reset is destructive, and the final task calls the model and writes a file.

### Review notes - logic left unchanged

1. The final cell calls `board.claim_todo(goal_id)`, which sets the goal to `in_progress`, but `INSTRUCTIONS` tells the worker to take a **pending** goal. The saved run completed, but these instructions disagree; aligning them would change the agent prompt and task-selection behavior.
2. The prompt says, “Your files live in the single your tools are allowed to use.” This is grammatically incomplete and may make the intended workspace restriction less clear to the model. The MCP server root still provides the filesystem boundary.
3. `plan_steps` returns a dictionary field spelled `steps_ids`; the related `pydantic_worker.py` spells this `step_ids`. The current run succeeds, but consistent spelling would make the tool result easier to consume reliably.
4. The Step 3 `board_agent` does not register `plan_steps`; it can inspect or complete tasks, but it cannot create a plan until Step 5. This is compatible with the example's status question, though the unused function may be surprising.
5. The final board reset drops existing rows. The resolved workspace path also depends on the notebook kernel's current working directory.

No executable logic was changed during this review. Existing saved outputs indicate the executed examples completed, but this review did not make external model calls, restart the MCP server, or rerun cells.